Procesado de datos de las baterías 13300 para el TFG de reciclado de baterías de vape.

Primero analizaremos los ciclos y veremos el SoH de las baterías

In [11]:
import os
import shutil
import re
import glob
from pathlib import Path


def organizar_archivos_por_fecha(carpeta_origen):
    """
    Lee los archivos de una carpeta, extrae la fecha de su nombre 
    y los mueve a subcarpetas organizadas por día.
    """
    
    # Expresión regular para buscar el formato de fecha AAAA_MM_DD
    # \d{4} = 4 dígitos (año), \d{2} = 2 dígitos (mes/día)
    patron_fecha = re.compile(r"(\d{4}_\d{2}_\d{2})")

    # Verificamos que la carpeta de origen exista
    if not os.path.exists(carpeta_origen):
        print(f"Error: La carpeta '{carpeta_origen}' no existe.")
        return

    # Iterar sobre todos los elementos en la carpeta
    for nombre_archivo in os.listdir(carpeta_origen):
        ruta_completa_origen = os.path.join(carpeta_origen, nombre_archivo)

        # Nos aseguramos de procesar solo archivos (ignoramos carpetas)
        if os.path.isfile(ruta_completa_origen):
            
            # Buscamos el patrón de la fecha en el nombre del archivo
            coincidencia = patron_fecha.search(nombre_archivo)
            
            if coincidencia:
                # Extraemos la fecha encontrada (ej. '2026_02_23')
                fecha_carpeta = coincidencia.group(1) 
                
                # Definimos la ruta de la nueva carpeta destino
                ruta_carpeta_destino = os.path.join("data/processed/cicladores", str(i), fecha_carpeta)
                
                # Si la carpeta de esa fecha no existe, la creamos
                if not os.path.exists(ruta_carpeta_destino):
                    os.makedirs(ruta_carpeta_destino)
                
                # Movemos el archivo
                ruta_completa_destino = os.path.join(ruta_carpeta_destino, nombre_archivo)
                shutil.copy(ruta_completa_origen, ruta_completa_destino)
                
                #print(f"Copiado: {nombre_archivo}  ->  Carpeta: {fecha_carpeta}/")
            else:
                print(f"Omitido (sin fecha detectada): {nombre_archivo}")



In [15]:


def trim_txt_files(processed_dir: str = "data/processed/cicladores") -> dict:
    """
    For every .txt file under data/processed/{1-6}/{dates}/:
      - Overwrites each file keeping only the header (line 1) and the last data row.
    For every .dat file in the same folders:
      - Deletes the file entirely.

    Parameters
    ----------
    processed_dir : str
        Path to the 'processed' folder, relative to the notebook location.
        Defaults to "data/processed".

    Returns
    -------
    dict with keys:
        "processed"  – list of .txt files successfully trimmed
        "skipped"    – list of .txt files skipped (< 2 lines or already trimmed)
        "deleted"    – list of .dat files deleted
        "errors"     – list of (filepath, error_message) tuples
    """
    base = Path(processed_dir)
    results = {"processed": [], "skipped": [], "deleted": [], "errors": []}

    def is_valid_folder(p: Path) -> bool:
        return p.parts[-3].isdigit() and 1 <= int(p.parts[-3]) <= 6

    # --- Process .txt files ---
    for filepath in sorted(base.glob("*/*/*.txt")):
        if not is_valid_folder(filepath):
            continue
        try:
            lines = filepath.read_text(encoding="latin-1").splitlines()

            if len(lines) < 2 or len(lines) == 2:
                results["skipped"].append(str(filepath))
                continue

            header   = lines[0]
            last_row = lines[-1]

            filepath.write_text(header + "\n" + last_row + "\n", encoding="latin-1")
            results["processed"].append(str(filepath))

        except Exception as e:
            results["errors"].append((str(filepath), str(e)))

    # --- Delete .dat files ---
    for filepath in sorted(base.glob("*/*/*.dat")):
        if not is_valid_folder(filepath):
            continue
        try:
            filepath.unlink()
            results["deleted"].append(str(filepath))
        except Exception as e:
            results["errors"].append((str(filepath), str(e)))

    # Summary
    print(f"✅ Trimmed : {len(results['processed'])} .txt files")
    print(f"⏭️  Skipped : {len(results['skipped'])} .txt files (already 2 lines or empty)")
    print(f"🗑️  Deleted : {len(results['deleted'])} .dat files")
    print(f"❌ Errors  : {len(results['errors'])} files")
    if results["errors"]:
        for fp, err in results["errors"]:
            print(f"   {fp}: {err}")

    return results

In [13]:

for i in range(1, 7):

    mi_carpeta = f"data/raw/cicladores/ciclador_{i}"
    
    print(f"Procesando: {mi_carpeta} ...")
    
    # Llamamos a la función con la ruta actualizada
    organizar_archivos_por_fecha(mi_carpeta)

print("¡Todas las carpetas han sido organizadas!")

Procesando: data/raw/cicladores/ciclador_1 ...
Procesando: data/raw/cicladores/ciclador_2 ...
Procesando: data/raw/cicladores/ciclador_3 ...
Procesando: data/raw/cicladores/ciclador_4 ...
Procesando: data/raw/cicladores/ciclador_5 ...
Procesando: data/raw/cicladores/ciclador_6 ...
¡Todas las carpetas han sido organizadas!


In [16]:

results = trim_txt_files() #porque ya conoce que tiene que buscar en data/processed


✅ Trimmed : 129 .txt files
⏭️  Skipped : 21 .txt files (already 2 lines or empty)
🗑️  Deleted : 150 .dat files
❌ Errors  : 0 files


Hemos limpiado el sistema de archivos: El resultado en la carpeta processed es una serie de archivos .txt en el que solamente se encuentra el último estado del ciclo que corresponde. Lo que nos interesa a continuación es casar cada archivo txt de Charge con la última carga correspondiente a cada batería.